# BG RL — آموزش تخته‌نرد در Google Colab

این notebook فقط برای آموزش و ارزیابی متنی است. رابط گرافیکی pygame را روی کامپیوتر Windows اجرا کنید.

checkpointها داخل Google Drive ذخیره می‌شوند تا با قطع شدن runtime از بین نروند.

In [ ]:
# تنظیمات اصلی
REPO_URL = 'https://github.com/Mooli-web/BG.git'
BRANCH = 'arena/01a0b49e-bg'
REPO_DIR = '/content/BG'

import os
import subprocess
import sys

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

%cd /content/BG
print('Repository ready:', os.getcwd())

In [ ]:
# نصب وابستگی‌های موردنیاز آموزش؛ pygame لازم نیست.
%pip install -q -r requirements-colab.txt

import numpy as np
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# اتصال Google Drive برای نگه‌داری checkpointها
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/BG_RL/checkpoints_fixed_rules'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints:', CHECKPOINT_DIR)

## آموزش از صفر

این اجرای جدید از checkpoint قبلی resume نمی‌کند؛ چون نسخه‌ی قبلی یک خطای قانونی در bearing off داشت. `total_steps` تعداد تصمیم‌های حرکت مهره است. برای شروع می‌توان ۵ میلیون step را اجرا کرد.

In [ ]:
import subprocess

train_command = [
    sys.executable, '-m', 'bg', 'train',
    '--total-steps', '5000000',
    '--num-envs', '16',
    '--rollout-steps', '256',
    '--device', 'auto',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--save-interval', '25',
    '--log-interval', '10',
    '--seed', '7',
]
print(' '.join(train_command))
subprocess.run(train_command, check=True)

## ادامه‌ی آموزش بعد از قطع شدن Colab

اگر runtime قطع شد، دوباره notebook را باز کنید، سلول‌های اتصال repository، نصب و Drive را اجرا کنید و سلول زیر را اجرا کنید. عدد `10000000` هدف نهایی است، نه تعداد step اضافه.

In [ ]:
resume_command = [
    sys.executable, '-m', 'bg', 'train',
    '--resume', os.path.join(CHECKPOINT_DIR, 'latest.pt'),
    '--total-steps', '10000000',
    '--num-envs', '16',
    '--rollout-steps', '256',
    '--device', 'auto',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--save-interval', '25',
    '--log-interval', '10',
]
subprocess.run(resume_command, check=True)

In [ ]:
# ارزیابی متنی در Colab، بدون pygame
evaluate_command = [
    sys.executable, '-m', 'bg', 'evaluate',
    '--checkpoint', os.path.join(CHECKPOINT_DIR, 'latest.pt'),
    '--games', '100',
    '--device', 'auto',
]
subprocess.run(evaluate_command, check=True)

In [ ]:
# اختیاری: دانلود checkpoint برای بازی روی Windows
from google.colab import files
files.download(os.path.join(CHECKPOINT_DIR, 'latest.pt'))